# Olikbochon — Bengali Hallucination Detector
## Colab Orchestrator

Run cells in order. This notebook:
1. Clones the repo
2. Installs dependencies
3. Mounts Google Drive (for persistent model weights)
4. Downloads datasets & builds FAISS index
5. Trains / evaluates / predicts

In [1]:
# Cell 1: Clone repo
import os
REPO_URL = "https://github.com/smabdullah2002/iut-datathon.git"
REPO_DIR = "/content/iut-datathon"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git lfs pull 2>/dev/null || true

Cloning into '/content/iut-datathon'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 32 (delta 4), reused 31 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 524.67 KiB | 5.00 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/iut-datathon
Already up to date.


In [ ]:
# Cell 2: Install dependencies
!pip install -r requirements.txt --quiet
print("Dependencies installed.")

In [ ]:
# Cell 3: Mount Google Drive (for persistent model weights)
from google.colab import drive
drive.mount('/content/drive')

# Create symlinks so scripts find the right paths
DRIVE_MODELS = "/content/drive/MyDrive/iut_datathon_models"
DRIVE_CORPUS = os.path.join(DRIVE_MODELS, "corpus")
os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(DRIVE_CORPUS, exist_ok=True)
print("Drive mounted. Model dir:", DRIVE_MODELS)

In [ ]:
# Cell 4: Download datasets & build retrieval index
# Takes ~10-30 min depending on download speed.
# Can be skipped on subsequent runs if index already exists.

import sys
sys.path.insert(0, "/content/iut-datathon")

from app.utils.download_data import (
    download_datasets,
    download_wikipedia,
    build_faiss_index,
)

download_datasets()
download_wikipedia()
build_faiss_index()

---
## Training & Inference

Run whichever cell applies to your current task.

In [ ]:
# Cell 5a: Train (with XNLI warmup, ~10 hrs)
# Saves best_model/ for use with 'predict' (Cell 5c). Not needed for 'meta' (Cell 5d/5e).
!python -m app.main train --epochs 10 --batch-size 16 --lr 2e-5

In [ ]:
# Cell 5a-retrieve: Train with retrieval augmentation (no XNLI, faster)
# Saves best_model/ for use with 'predict' (Cell 5c). Not needed for 'meta'.
!python -m app.main train --skip-xnli --epochs 10 --batch-size 16 --lr 2e-5 --retrieve

In [ ]:
# Cell 5b: C-Band annotation (one-time setup)
# Generates app/dataset/cband_annotation.csv with K-Means clusters.
# --auto uses keyword heuristics to pre-fill bands per cluster.
# Open the CSV, review each cluster, fix any misfires, save.
# After saving, all meta cells automatically use bands for stratification & threshold tuning.
!python -m app.main annotate --auto

import pandas as pd
from IPython.display import display, HTML
df = pd.read_csv("app/dataset/cband_annotation.csv")
display(HTML(f"""
<b>Annotation saved.</b><br>
{len(df)} rows in {df['cluster'].nunique()} clusters.<br>
Band distribution: {df['band'].value_counts().to_dict()}<br><br>
<ol>
  <li>Download <code>app/dataset/cband_annotation.csv</code></li>
  <li>Open in any spreadsheet editor</li>
  <li>Review the <b>band</b> column — fix any misfires (most will be correct)</li>
  <li>Upload back to <code>app/dataset/cband_annotation.csv</code></li>
  <li>Re-run any meta cell below — bands load automatically</li>
</ol>
"""))

In [ ]:
# Cell 5c: Predict using best_model/ + retrieval (baseline Track A+B)
# Uses best_model/ from Cell 5a or 5a-retrieve. Submit output to Kaggle.
!python -m app.main predict --output submission_retrieval.csv --retrieve

In [ ]:
# Cell 5d: Meta-classifier — Single seed, fast (no XNLI, ~30 min on T4)
# Trains BanglaBERT via 5-fold CV internally (no data leakage), trains final model
# on all data, extracts test features, then trains LightGBM. No dependency on best_model/.
# Useful for quick iteration before running the full ensemble.
!python -m app.main meta --skip-xnli --retrieve --epochs 10 --batch-size 8 --seeds 1

In [ ]:
# Cell 5e: Meta-classifier — 3-seed ensemble, fast (no XNLI, ~90 min on T4)
# Averages OOF probabilities and test probabilities across 3 seeds (42, 43, 44).
# Trains a single LightGBM on averaged features. Usually +0.02-0.04 F1 over single seed.
!python -m app.main meta --skip-xnli --retrieve --epochs 10 --batch-size 8 --seeds 3

In [ ]:
# Cell 5f: Meta-classifier — 3-seed ensemble, full (with XNLI warmup, ~3-4 hrs on T4)
# Same as 5e but starts with XNLI warmup (381k Bengali NLI pairs) before OOF CV.
# Best configuration when you want the final submission.
!python -m app.main meta --retrieve --epochs 10 --batch-size 8 --seeds 3

In [ ]:
# Cell 5g: Threshold-tuned encoder — No LightGBM (fast, no stacking overhead)
# Trains BanglaBERT with early stopping, searches optimal threshold on val set,
# applies to test set. No OOF, no LightGBM, no feature engineering.
# Try this if meta (5d/5e) gives worse scores than baseline (5c).
!python -m app.main threshold --skip-xnli --retrieve --epochs 30 --batch-size 16 --seeds 3